In [1]:
# Imports
import torch

In [2]:
# Model analog thermometer which does not show units
# Temperature in celcius
t_celcius = [0.5, 14.0, 15.0, 28.0, 11.0, 8.0, 3.0, -4.0, 6.0, 13.0, 21.0]

# Unknown Temperature units
t_unk = [35.7, 55.9, 58.2, 81.9, 56.3, 48.9, 33.9, 21.8, 48.4, 60.4, 68.4]


t_celcius= torch.tensor(t_celcius)
t_unk = torch.tensor(t_unk)


In [3]:
print(t_celcius)
print(t_unk)

tensor([ 0.5000, 14.0000, 15.0000, 28.0000, 11.0000,  8.0000,  3.0000, -4.0000,
         6.0000, 13.0000, 21.0000])
tensor([35.7000, 55.9000, 58.2000, 81.9000, 56.3000, 48.9000, 33.9000, 21.8000,
        48.4000, 60.4000, 68.4000])


In [4]:
# Model
def model(t_unk, w, b):
    return w * t_unk + b

def loss_function(t_prediction, t_celcius):
    # Mean squared error
    squared_difference = (t_prediction - t_celcius) ** 2
    return squared_difference.mean()

In [5]:
w = torch.ones(())
b = torch.zeros(())

print(w)
print(b)

tensor(1.)
tensor(0.)


In [6]:
# Fitting the model
t_predictions = model(t_unk, w , b)
t_predictions

tensor([35.7000, 55.9000, 58.2000, 81.9000, 56.3000, 48.9000, 33.9000, 21.8000,
        48.4000, 60.4000, 68.4000])

In [7]:
loss = loss_function(t_predictions, t_celcius)

In [8]:
print(loss)

tensor(1763.8848)


In [9]:
# Decreasing the loss
# The rate of change of the loss with respect to each parameter
#If i nudge the weights and bias how does the loss function respond, does it go up or down
delta = 0.1
loss_rate_of_change_w = \
    (loss_function(model(t_unk, w + delta, b), t_celcius) - 
     loss_function(model(t_unk, w - delta, b), t_celcius)) / (2.0 * delta)

loss_rate_of_change_b = \
    (loss_function(model(t_unk, w, b + delta), t_celcius) -
     loss_function(model(t_unk, w, b - delta), t_celcius)) / (2.0 * delta)


In [10]:
# Scaling factor , step size to change the parameters slowly
learning_rate = 1e-2

#w = w - learning_rate * loss_rate_of_change_w
#b = b - learning_rate * loss_rate_of_change_b



In [11]:
# This approach does not scale with many parameters, we need to reuse calculations to find adjustments in latter layers
# d loss_fn / d w = (d loss_fn / d t_p) * (d t_p / d w)

# Derivative of loss function

def dloss_function(t_predictions , t_celcius):
    d_squared_diff = 2 * (t_predictions - t_celcius) / t_predictions.size(0)
    return d_squared_diff

In [12]:
def dmodel_dw(t_unk , w, b):
    return t_unk

def dmodel_db(t_unk , w, b):
    return 1.0


In [15]:
# Gradient function : Gradient of loss with respect with w and b
def gradient_function(t_unk, t_celcius, t_predictions, w , b):
    derivative_loss_fn      = dloss_function(t_predictions, t_celcius)
    derivative_loss_weights = derivative_loss_fn * dmodel_dw(t_unk, w, b)
    derivative_loss_bias    = derivative_loss_fn * dmodel_db(t_unk, w, b)

    return torch.stack([derivative_loss_weights.sum(), derivative_loss_bias.sum()])

In [16]:
grad = gradient_function(t_unk, t_celcius, t_predictions, w, b)
grad

tensor([4517.2964,   82.6000])

In [26]:
def trainining_loop(n_epochs, learning_rate, params, t_unk, t_celcius):
    for epoch in range(1, n_epochs + 1):
        w, b = params

        t_pred = model(t_unk, w, b)
        loss = loss_function(t_pred, t_celcius)
        grad = gradient_function(t_unk, t_celcius,t_predictions, w , b)

        params = params - learning_rate * grad 
        print(f'Epoch: {epoch}, Loss: {float(loss)}')

    return params



In [31]:
trainining_loop(
    n_epochs = 100,
    learning_rate = 1e-2,
    params = torch.tensor([1.0, 0.0]),
    t_unk = t_unk,
    t_celcius = t_celcius
)

Epoch: 1, Loss: 1763.884765625
Epoch: 2, Loss: 5802484.5
Epoch: 3, Loss: 23612904.0
Epoch: 4, Loss: 53433024.0
Epoch: 5, Loss: 95262832.0
Epoch: 6, Loss: 149102336.0
Epoch: 7, Loss: 214951568.0
Epoch: 8, Loss: 292810496.0
Epoch: 9, Loss: 382679136.0
Epoch: 10, Loss: 484557504.0
Epoch: 11, Loss: 598445504.0
Epoch: 12, Loss: 724343360.0
Epoch: 13, Loss: 862250752.0
Epoch: 14, Loss: 1012167936.0
Epoch: 15, Loss: 1174094720.0
Epoch: 16, Loss: 1348031360.0
Epoch: 17, Loss: 1533977600.0
Epoch: 18, Loss: 1731933696.0
Epoch: 19, Loss: 1941899264.0
Epoch: 20, Loss: 2163874816.0
Epoch: 21, Loss: 2397859584.0
Epoch: 22, Loss: 2643854336.0
Epoch: 23, Loss: 2901858816.0
Epoch: 24, Loss: 3171872768.0
Epoch: 25, Loss: 3453896448.0
Epoch: 26, Loss: 3747929856.0
Epoch: 27, Loss: 4053974016.0
Epoch: 28, Loss: 4372026880.0
Epoch: 29, Loss: 4702088704.0
Epoch: 30, Loss: 5044161024.0
Epoch: 31, Loss: 5398243840.0
Epoch: 32, Loss: 5764336128.0
Epoch: 33, Loss: 6142436864.0
Epoch: 34, Loss: 6532548608.0
Epoc

tensor([-4516.2905,   -82.5999])

In [30]:
trainining_loop(
    n_epochs = 100,
    learning_rate = 1e-4,
    params = torch.tensor([1.0, 0.0]),
    t_unk = t_unk,
    t_celcius = t_celcius
)

Epoch: 1, Loss: 1763.884765625
Epoch: 2, Loss: 323.09051513671875
Epoch: 3, Loss: 83.26638793945312
Epoch: 4, Loss: 1044.4122314453125
Epoch: 5, Loss: 3206.52734375
Epoch: 6, Loss: 6569.61279296875
Epoch: 7, Loss: 11133.66796875
Epoch: 8, Loss: 16898.6953125
Epoch: 9, Loss: 23864.6875
Epoch: 10, Loss: 32031.6484375
Epoch: 11, Loss: 41399.58203125
Epoch: 12, Loss: 51968.484375
Epoch: 13, Loss: 63738.36328125
Epoch: 14, Loss: 76709.2109375
Epoch: 15, Loss: 90881.0234375
Epoch: 16, Loss: 106253.8046875
Epoch: 17, Loss: 122827.59375
Epoch: 18, Loss: 140602.3125
Epoch: 19, Loss: 159578.0
Epoch: 20, Loss: 179754.6875
Epoch: 21, Loss: 201132.3125
Epoch: 22, Loss: 223710.953125
Epoch: 23, Loss: 247490.5
Epoch: 24, Loss: 272471.125
Epoch: 25, Loss: 298652.59375
Epoch: 26, Loss: 326035.09375
Epoch: 27, Loss: 354618.59375
Epoch: 28, Loss: 384403.0
Epoch: 29, Loss: 415388.375
Epoch: 30, Loss: 447574.8125
Epoch: 31, Loss: 480962.125
Epoch: 32, Loss: 515550.46875
Epoch: 33, Loss: 551339.75
Epoch: 34

tensor([-44.1729,  -0.8260])